# EGMS API demo

We start by setting up some imports. You can install `requests` and `jwt` as follows:

    pip3 install requests
    pip3 install pyjwt


In [ ]:
import sys
import json
import time
import requests
import jwt

## Access token

First, you must obtain an API token from the CLMS website. Follow the steps outlined in the [CLMS API documentation](https://eea.github.io/clms-api-docs/authentication.html#login-in-the-clms-website). We will use this API token to obtain a short-lived access token which will grant us access to the EGMS API as well. In the example below, the token is loaded from a file called `token.jwt` located in the same directory as this notebook.

In [ ]:
def get_access_token():
    service_key = json.load(open('token.jwt', 'rb'))
    private_key = service_key['private_key'].encode('utf-8')
    claim_set = {
        "iss": service_key['client_id'],
        "sub": service_key['user_id'],
        "aud": service_key['token_uri'],
        "iat": int(time.time()),
        "exp": int(time.time() + (60 * 60)),
    }
    grant = jwt.encode(claim_set, private_key, algorithm='RS256')
    result = requests.post(service_key["token_uri"], headers={ "Accept": "application/json", "Content-Type": "application/x-www-form-urlencoded" },
            data={ "grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer", "assertion": grant } )
    access_token_info_json = result.json()
    access_token = access_token_info_json.get('access_token')
    return access_token

#### Obtain access token and configure headers

We use the get_access_token() function to obtain the access token and then configure our request headers to include it. 

In [ ]:
access_token = get_access_token()
headers = {"Authorization" : f"Bearer {access_token}", "Accept" : "application/json"}

## EGMS API query

Let's build some queries to the API and post them. A query is simply a JSON object containing a set of keys that filter the result set on the server side. The following keys are supported:

- `levels` : **[Required]** Only products from the supplied list of levels are returned
- `releases` : **[Required]** Only products from the specified list of release(s) are returned
- `bbox` : Only products overlapping the bounding box are returned. The bounding box is specified as a list of lon, lat coordinates:
    - `[ [min_lon, min_lat], [max_lon, max_lat] ]`.
    - The maximum size of the bounding box is 5 degrees in longitude and latitude.
    - If you specify more than two points, the bounding box is computed as the minimum and maximum longitudes and latitudes of all the supplied points. It is also possible to provide only a single point.
- `swath` : Only products with the given swath are returned.
    - Example: `IW1`
- `burstCycle` or `burst_cycle` : Only products with the given burst cycle are returned
- `relativeOrbit` or `relative_orbit` : Only products with the given relative orbit are returned
- `direction` : Only products with the given direction (ascending or descending) are returned.
- `tileId` : Only products with the given tile ID are returned.
- `productType` : Only products with the given product type are returned. This filter is usually not needed and is already provided through the `levels` key. However, it may be used to distinguish L3-datasets between `ORTHO-UP` and `ORTHO-EAST`.
- `id` : If you already have a query ID from a previous query, you may reuse the same query ID here. Note that the query ID returned by the API may change at any time. The query ID is used to download data later, and expires if it is not used to download data.

A list of valid values for levels, releases, swaths, burst cycles, relative orbits, tile IDs, product types and direction can be obtained by calling the respective API endpoints, as illustrated below.

In [ ]:
#api_endpoint = "http://localhost:2008/archive"
api_endpoint = "https://egms.land.copernicus.eu/insar-api/archive"

# Get a list of levels, product releases, swaths, burst cycles, relative orbits and directions
info = { "levels" : requests.get(f"{api_endpoint}/levels", headers=headers).json(),
         "releases" : requests.get(f"{api_endpoint}/releases", headers=headers).json(),
         "swaths" : requests.get(f"{api_endpoint}/swaths", headers=headers).json(),
         "relative_orbits" : requests.get(f"{api_endpoint}/relative_orbits", headers=headers).json(),
         "bursts" : requests.get(f"{api_endpoint}/bursts", headers=headers).json(),
         "directions" : requests.get(f"{api_endpoint}/directions", headers=headers).json(),
         "tile_ids" : requests.get(f"{api_endpoint}/tile_ids", headers=headers).json(),
         "product_types" : requests.get(f"{api_endpoint}/product_types", headers=headers).json()
       }

for key, value in info.items():
    if type(value) == type({}):
        print(f"Error: {value}")
    else:
        print(f"Available {key}: {value[0:5]} {'...' if len(value) > 5 else ''}")


### Query types

Queries generally come in two flavors: A query with a bounding box and optional filters, or a query specifying either a burst cycle, relative orbit, swath or a level 3 tile ID. A bounding box query may additionally include any of the filters for burst cycle, relative orbit, swath, direction, tile ID or product type. Remember to always include the levels and releases filters. The filters primarily apply to L2A and L2B products. Queries for L3 products require either a tile ID or a bounding box.

In [ ]:
# Let's make a query for datasets within the bounding box below.
# Note that we can supply the bounding box as a list of lon, lat pairs, and the server
# will build the bounding box based on the minimum and maximum values of the supplied points.
query = {"id": None,
         "bbox": [[3.5652673514508684, 50.52992617658616],
                   [3.6410148180443005, 48.99586980500084],
                   [5.553848001377699, 48.96112650494832],
                   [5.655334370641719, 50.59849181002919]],
         "levels" : ["L3"],
         "releases" : ["2019-2023"]
         }

r = requests.post(f"{api_endpoint}/search", headers=headers, data=json.dumps(query))
result = r.json()
result

### Query without bounding box

Let's try making a query without a bounding box.

In [ ]:
query = {"id": None,
         "swath" : "IW1",
         "burstCycle" : "0840",
         "direction" : "descending",
         "releases" : ["2019-2023"],
         "levels" : ["L2B"]
         }
r = requests.post(f"{api_endpoint}/search", headers=headers, data=json.dumps(query))
result = r.json()
print(result["message"]) if "message" in result else None
print(f"Got {len(result['hits'])} hits. First hit:")
result["hits"][0] if len(result["hits"]) > 0 else None


In [ ]:
# Let's run a query for a single L3 dataset
query = {"id": None,
         "tileId" : "E40N30",
         "releases" : ["2019-2023"],
         "levels" : ["L3"],
         "productType" : "ORTHO-UP"
         }
r = requests.post(f"{api_endpoint}/search", headers=headers, data=json.dumps(query))
result = r.json()
result

## Query results

All queries return a JSON object containing the query results. The general format of the result is as follows:

- `status` : A boolean indicating whether the query was successful or not
- `hits` : A possibly empty list containing the query results
- `id` : A string that is used when constructing download links
- `message` : *[Optional]* Typically only provided for queries that fail. May describe what you must do to make the query succeed.

### Hits

Each object in the `hits` list contains a description of a single dataset. All datasets have the following keys:

- `filename` : The filename of the dataset. This is used when generating links to download the dataset.
- `filesize` : The size of the (compressed) dataset in bytes.
- `bbox` : The bounding box of the dataset. Each item in the list is a `[lon, lat]` pair of coordinates.
- `poly` : A polygon describing the area that the dataset actually covers. The polygon always covers a smaller area than the bounding box.
- `productLevel` : The product level of the dataset (L2A, L2B or L3)
- `productType` : The product type. Currently either BASIC, CALIBRATED, ORTHO-EAST or ORTHO-UP.
- `release` : The release this dataset belongs to.
- `sub_release` : An integer that increments whenever a dataset is updated.
- `version` : The dataset version. Obtained by concatenating `release` with `sub_release`.

In addition, datasets may have some of the following properties, depending on product type: `burstId`, `relativeOrbit`, `burstCycle`, `swath`, `polarization`, `direction` and `tileId`. The `burstId` property is a combination of `relativeOrbit`, `burstCycle`, `swath` and `polarization`.

## Downloading datasets

To download a dataset identified by the query results, you must construct a link to the dataset. The link is constructed as follows:

`api_endpoint` + `/download/` + `filename` + `?id=` + `id`

The `filename` is obtained from a dataset description in the list of hits from the query result, and the `id` is obtained from the `id` field of the query result. Here's an example:

In [ ]:
links = []
for hit in result["hits"]:
    link = f"{api_endpoint}/download/{hit['filename']}?id={result['id']}"
    links.append(link)
print(links)

The resulting links can be used to download datasets by calling an external tool such as `curl` or `wget` or by implementing your own download code in your language of choice. Note that if the query ID is idle (i.e, not used to download anything or perform additional queries), the ID will eventually expire and the query must be performed again to obtain a new query ID.